# Zajęcia 1 — Sekwencje i sygnał mowy
Log-mel i MFCC dla trzech własnych nagrań (samogłoska, wyraz, zdanie) oraz wpływ `n_fft` i `hop_length`.

In [ ]:
!pip -q install librosa soundfile matplotlib numpy scipy

In [ ]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

SR = 16000       # częstotliwość próbkowania [Hz]
N_FFT = 512      # długość okna (32 ms)
HOP = 160        # krok ramkowania (10 ms)
N_MELS = 64      # liczba pasm mel
N_MFCC = 13      # liczba współczynników MFCC
N_FFT_LIST = [256, 512, 1024]   # eksperyment A
HOP_LIST = [80, 160, 320]       # eksperyment B

os.makedirs("wykresy", exist_ok=True)

## 1. Wczytanie nagrań

In [ ]:
from google.colab import files
uploaded = files.upload()
PLIKI = list(uploaded.keys())
print(PLIKI)

In [ ]:
# Numery plików z listy powyżej: samogłoska, wyraz, zdanie
NAGRANIA = {"samogloska": PLIKI[0], "wyraz": PLIKI[1], "zdanie": PLIKI[2]}
ETYKIETY = {"samogloska": "samogłoska", "wyraz": "wyraz", "zdanie": "zdanie"}

SYG = {}
for nazwa, plik in NAGRANIA.items():
    # resampling do 16 kHz i konwersja do mono
    SYG[nazwa], _ = librosa.load(plik, sr=SR, mono=True)
    y = SYG[nazwa]
    print(f"{nazwa:11s} {len(y):6d} próbek, {len(y) / SR:.2f} s, "
          f"peak={np.max(np.abs(y)):.3f}, RMS={np.sqrt(np.mean(y ** 2)):.4f}")

## 2. Funkcje: log-mel i MFCC

In [ ]:
def licz_logmel(y, n_fft, hop):
    """Log-mel w dB względem maksimum nagrania (0 dB = maksimum)."""
    S = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=n_fft, win_length=n_fft, hop_length=hop, n_mels=N_MELS
    )
    return librosa.power_to_db(S, ref=np.max)


def licz_mfcc(S_db):
    """MFCC liczone z tej samej macierzy co log-mel (DCT-II)."""
    return librosa.feature.mfcc(S=S_db, n_mfcc=N_MFCC)


def zakres_mfcc(M):
    """Symetryczna skala barw dobrana do współczynników 1-12 (c0 jest poza skalą)."""
    return np.percentile(np.abs(M[1:]), 98)


LOGMEL = {n: licz_logmel(y, N_FFT, HOP) for n, y in SYG.items()}
MFCC = {n: licz_mfcc(L) for n, L in LOGMEL.items()}

print(f"{'nagranie':11s} {'próbki':>7s} {'log-mel':>10s} {'MFCC':>9s} {'T (wzór)':>9s}")
for n, y in SYG.items():
    t_wzor = 1 + len(y) // HOP   # center=True
    print(f"{n:11s} {len(y):7d} {str(LOGMEL[n].shape):>10s} {str(MFCC[n].shape):>9s} {t_wzor:9d}")

## 3. Przebieg, log-mel i MFCC każdego nagrania

In [ ]:
for n, y in SYG.items():
    L, M = LOGMEL[n], MFCC[n]
    fig, ax = plt.subplots(3, 1, figsize=(10, 9))

    librosa.display.waveshow(y, sr=SR, ax=ax[0])
    ax[0].set_title(f"{ETYKIETY[n]} - przebieg czasowy")
    ax[0].set_ylabel("amplituda")

    im1 = librosa.display.specshow(L, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel",
                                   fmax=SR / 2, vmin=-80, vmax=0, ax=ax[1])
    ax[1].set_title(f"{ETYKIETY[n]} - log-mel ({N_MELS} pasm, n_fft={N_FFT}, hop={HOP})")
    fig.colorbar(im1, ax=ax[1], format="%+2.0f dB")

    lim = zakres_mfcc(M)
    im2 = librosa.display.specshow(M, sr=SR, hop_length=HOP, x_axis="time",
                                   vmin=-lim, vmax=lim, cmap="coolwarm", ax=ax[2])
    ax[2].set_title(f"{ETYKIETY[n]} - MFCC ({N_MFCC} współczynników)")
    ax[2].set_ylabel("numer współczynnika")
    fig.colorbar(im2, ax=ax[2], label="wartość współczynnika")

    for a in ax:                       # ten sam zakres czasu na wszystkich wykresach
        a.set_xlim(0, len(y) / SR)
        a.set_xlabel("czas [s]")
    fig.tight_layout()
    fig.savefig(f"wykresy/01_{n}_przebieg_logmel_mfcc.png", bbox_inches="tight")
    plt.show()

## 4. Porównanie nagrań (wspólna skala dB)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 7), layout="constrained")
for j, n in enumerate(SYG):
    im = librosa.display.specshow(LOGMEL[n], sr=SR, hop_length=HOP, x_axis="time",
                                  y_axis="mel", fmax=SR / 2, vmin=-80, vmax=0, ax=ax[0, j])
    ax[0, j].set_title(f"log-mel: {ETYKIETY[n]}")
    lim = zakres_mfcc(MFCC[n])
    librosa.display.specshow(MFCC[n], sr=SR, hop_length=HOP, x_axis="time",
                             vmin=-lim, vmax=lim, cmap="coolwarm", ax=ax[1, j])
    ax[1, j].set_title(f"MFCC: {ETYKIETY[n]}")
    ax[1, j].set_ylabel("numer współczynnika")
fig.colorbar(im, ax=ax[0, :].tolist(), format="%+2.0f dB", shrink=0.8)
fig.savefig("wykresy/02_porownanie_nagran.png", bbox_inches="tight")
plt.show()

## 5. Eksperyment A: długość okna (`n_fft`), stały `hop_length = 160`

In [ ]:
# log-mel: 3 nagrania x 3 wartości n_fft
fig, ax = plt.subplots(3, 3, figsize=(15, 10), layout="constrained")
for i, n in enumerate(SYG):
    for j, nf in enumerate(N_FFT_LIST):
        L = licz_logmel(SYG[n], nf, HOP)
        im = librosa.display.specshow(L, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel",
                                      fmax=SR / 2, vmin=-80, vmax=0, ax=ax[i, j])
        ax[i, j].set_title(f"{ETYKIETY[n]}, n_fft={nf} ({1000 * nf / SR:.0f} ms)")
fig.colorbar(im, ax=ax.ravel().tolist(), format="%+2.0f dB", shrink=0.6)
fig.savefig("wykresy/03_ekspA_logmel_wszystkie.png", bbox_inches="tight")
plt.show()

In [ ]:
# STFT zdania (0-4 kHz): tu najlepiej widać harmoniczne
WYBRANE = "zdanie"
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True, layout="constrained")
for j, nf in enumerate(N_FFT_LIST):
    D = librosa.stft(SYG[WYBRANE], n_fft=nf, win_length=nf, hop_length=HOP)
    Sdb = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    im = librosa.display.specshow(Sdb, sr=SR, hop_length=HOP, x_axis="time", y_axis="hz",
                                  vmin=-80, vmax=0, ax=ax[j])
    ax[j].set_ylim(0, 4000)
    ax[j].set_title(f"{WYBRANE}: STFT, n_fft={nf} ({1000 * nf / SR:.0f} ms)")
fig.colorbar(im, ax=ax.tolist(), format="%+2.0f dB", shrink=0.8)
fig.savefig("wykresy/04_ekspA_stft_wybrane.png", bbox_inches="tight")
plt.show()

# MFCC zdania przy różnych n_fft
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True, layout="constrained")
for j, nf in enumerate(N_FFT_LIST):
    M = licz_mfcc(licz_logmel(SYG[WYBRANE], nf, HOP))
    lim = zakres_mfcc(M)
    im = librosa.display.specshow(M, sr=SR, hop_length=HOP, x_axis="time",
                                  vmin=-lim, vmax=lim, cmap="coolwarm", ax=ax[j])
    ax[j].set_title(f"{WYBRANE}: MFCC, n_fft={nf}")
    ax[j].set_ylabel("numer współczynnika")
fig.colorbar(im, ax=ax.tolist(), shrink=0.8)
fig.savefig("wykresy/05_ekspA_mfcc_wybrane.png", bbox_inches="tight")
plt.show()

## 6. Eksperyment B: krok (`hop_length`), stałe `n_fft = 512`

In [ ]:
print(f"{'nagranie':11s} {'hop':>5s} {'T zmierzone':>12s} {'T wzór':>8s}")
for n, y in SYG.items():
    for h in HOP_LIST:
        T = licz_logmel(y, N_FFT, h).shape[1]
        print(f"{n:11s} {h:5d} {T:12d} {1 + len(y) // h:8d}")

fig, ax = plt.subplots(3, 3, figsize=(15, 10), layout="constrained")
for i, n in enumerate(SYG):
    for j, h in enumerate(HOP_LIST):
        L = licz_logmel(SYG[n], N_FFT, h)
        im = librosa.display.specshow(L, sr=SR, hop_length=h, x_axis="time", y_axis="mel",
                                      fmax=SR / 2, vmin=-80, vmax=0, ax=ax[i, j])
        ax[i, j].set_title(f"{ETYKIETY[n]}, hop={h} ({1000 * h / SR:.0f} ms), T={L.shape[1]}")
fig.colorbar(im, ax=ax.ravel().tolist(), format="%+2.0f dB", shrink=0.6)
fig.savefig("wykresy/06_ekspB_logmel_wszystkie.png", bbox_inches="tight")
plt.show()

In [ ]:
# zbliżenie 0,3-0,6 s zdania: gęstość ramek przy różnym kroku
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True, layout="constrained")
for j, h in enumerate(HOP_LIST):
    L = licz_logmel(SYG[WYBRANE], N_FFT, h)
    im = librosa.display.specshow(L, sr=SR, hop_length=h, x_axis="time", y_axis="mel",
                                  fmax=SR / 2, vmin=-80, vmax=0, ax=ax[j])
    ax[j].set_xlim(0.3, 0.6)
    ax[j].set_title(f"{WYBRANE}: hop={h}, zakres 0,3-0,6 s")
fig.colorbar(im, ax=ax.tolist(), format="%+2.0f dB", shrink=0.8)
fig.savefig("wykresy/07_ekspB_zblizenie.png", bbox_inches="tight")
plt.show()

## 7. Pomiary do analizy w sprawozdaniu

In [ ]:
# f0 (pYIN), udział energii poniżej 1 kHz i zmienność MFCC w czasie
print(f"{'nagranie':11s} {'f0 mediana':>11s} {'f0 min-max':>13s} {'energia<1k':>11s} "
      f"{'std c1':>7s} {'std c1-12 max':>14s}")
for n, y in SYG.items():
    f0, _, _ = librosa.pyin(y, fmin=70, fmax=400, sr=SR, frame_length=1024, hop_length=HOP)
    D = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)) ** 2
    fr = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
    udzial = D[fr < 1000].sum() / D.sum()
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)[0]
    akt = rms > 0.02                                  # ramki aktywne
    std = MFCC[n][1:, akt].std(axis=1)                # c1..c12
    print(f"{n:11s} {np.nanmedian(f0):9.1f} Hz {np.nanmin(f0):5.0f}-{np.nanmax(f0):4.0f} Hz "
          f"{udzial:10.0%} {std[0]:7.0f} {std.max():14.0f}")

In [ ]:
# kontrast harmonicznych samogłoski (100-1000 Hz, 0,3-1,0 s) w zależności od n_fft:
# odchylenie widma w dB od jego wygładzonej obwiedni (większe = wyraźniejsze prążki)
y = SYG["samogloska"]
for nf in N_FFT_LIST:
    D = np.abs(librosa.stft(y, n_fft=nf, win_length=nf, hop_length=HOP)) ** 2
    fr = librosa.fft_frequencies(sr=SR, n_fft=nf)
    S = D[:, 30:100].mean(axis=1)
    s = 10 * np.log10(S[(fr >= 100) & (fr <= 1000)] + 1e-12)
    obw = median_filter(s, size=max(3, int(round(300 / (SR / nf)))))
    print(f"n_fft={nf:5d} (okno {1000 * nf / SR:3.0f} ms): kontrast = {np.std(s - obw):.1f} dB")

# poziom w pasmach mel samogłoski (artefakt przy 6,5-7,2 kHz)
fm = librosa.mel_frequencies(N_MELS, fmin=0, fmax=SR / 2)
sred = LOGMEL["samogloska"][:, 20:120].mean(axis=1)
for lo, hi in [(5500, 6500), (6500, 7200), (7200, 8000)]:
    print(f"pasmo {lo}-{hi} Hz: {sred[(fm >= lo) & (fm < hi)].mean():.1f} dB")

In [ ]:
# cechy czasowe wybranego nagrania (do opisu zdarzeń: szum = wysoki centroid i ZCR, pauza = niski RMS)
NAGRANIE_CECHY = "zdanie"          # zmień na "wyraz", aby wypisać cechy wyrazu
y = SYG[NAGRANIE_CECHY]
rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)[0]
cen = librosa.feature.spectral_centroid(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP)[0]
zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP)[0]
f0, _, _ = librosa.pyin(y, fmin=70, fmax=400, sr=SR, frame_length=1024, hop_length=HOP)

print("t [s]   RMS   centroid [Hz]   ZCR    f0 [Hz]")
for i in range(0, len(rms), 4):                       # co 40 ms
    f = "-" if np.isnan(f0[i]) else f"{f0[i]:.0f}"
    print(f"{i * HOP / SR:5.2f} {rms[i]:7.3f} {cen[i]:11.0f} {zcr[i]:8.2f} {f:>8s}")

In [ ]:
import shutil
shutil.make_archive("wykresy", "zip", "wykresy")
files.download("wykresy.zip")